# Time- and frequency-domain analysis

Stimulus and sway-response time signals and amplitude spectra for no stimulus and velocities of 0.01, 0.1, 1, and 10 °/s. Group results pool recordings and cycles across participants, as in `2a_SmallRampAnalysis.ipynb`. The stimulation frequency is 0.5 Hz.


In [1]:
import sys
from pathlib import Path

search_roots = [Path.cwd(), *Path.cwd().parents]
project_root = next(
    (root for root in search_roots if (root / 'src' / 'config.py').is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError('Could not find the SmallRamp_Analysis project root.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import balancepy as bp
import balancepy.anaropia as bpa
import balancepy.anaropia_project as bpap

from src.config import paths

# Analysis settings

Use the same conditions and preprocessing as `2a_SmallRampAnalysis.ipynb`.


In [2]:
MAIN_FREQUENCY_HZ = 0.5
CONDITIONS = [
    {'label': 'No stim', 'condition_ids': [1, 13], 'velocity': None},
    {'label': '0.01', 'condition_ids': [2, 3, 14], 'velocity': 0.01},
    {'label': '0.1', 'condition_ids': [20], 'velocity': 0.1},
    {'label': '1', 'condition_ids': [9], 'velocity': 1.0},
    {'label': '10', 'condition_ids': [11], 'velocity': 10.0},
]

preproc_config = bpa.AnaropiaPreprocessingConfig()
preproc_config.end_time_seconds = 210
preproc_config.cut_to_cycles = True
preproc_config.cycle_length_samples = 180
preproc_config.cycle_start_samples = 878

sr_config = bpa.AnaropiaSRDataConfig(
    stimulus_name='stim_pitch',
    response_name=bpa.COM_LEGACY_AP,
    frequency_selection='all',
)

# Load and preprocess data across all subjects

Merge recordings within each condition to calculate pooled time signals and spectra.


In [3]:
condition_results = []
all_records = []

for specification in CONDITIONS:
    recordings = []
    participant_ids = set()
    for subject_id, condition, filepath, body_height_m in bpap.batch_iterator(
        paths=paths,
        conditions=specification['condition_ids'],
        output_columns=['body_height_m'],
    ):
        sr = bpa.build_sr_data(
            filename=filepath,
            body_height_m=body_height_m,
            sr_config=sr_config,
            preproc_config=preproc_config,
            name=f'{subject_id}_{condition}',
        )
        record = {
            'subject_id': str(subject_id),
            'condition': condition,
            'filepath': Path(filepath),
            'body_height_m': body_height_m,
            'sr': sr,
        }
        recordings.append(sr)
        participant_ids.add(str(subject_id))
        all_records.append({**record, 'condition_label': specification['label']})

    if not recordings:
        raise RuntimeError(f"No usable recordings found for {specification['label']}")

    group = bp.merge_sr_data(recordings, name=specification['label'])
    condition_results.append({
        **specification,
        'group': group,
        'n_participants': len(participant_ids),
    })

pd.DataFrame([
    {
        'condition': result['label'],
        'condition_ids': result['condition_ids'],
        'participants': result['n_participants'],
    }
    for result in condition_results
])

,condition,condition_ids,participants
0,No stim,"[1, 13]",13
1,0.01,"[2, 3, 14]",13
2,0.1,[20],12
3,1,[9],13
4,10,[11],13


# Amplitude spectra for one selected trial

Participant `AB12OC03`, trial `t19`, at 0.1 °/s. Grey markers show individual cleaned cycles; blue and green stems and markers show the magnitudes of the mean complex stimulus and response spectra. The stimulation frequency is 0.5 Hz.


In [4]:
example_candidates = [
    record for record in all_records
    if record['condition_label'] == '0.1'
    and record['subject_id'] == 'AB12OC03'
    and record['filepath'].stem.endswith('_t19')
]
if not example_candidates:
    raise RuntimeError('Could not find usable AB12OC03, condition 0.1, trial t19 data.')
example = example_candidates[0]

# Use the results preprocessing: 180 samples (2 seconds) per cycle, starting
# at sample 878. This gives frequency points spaced by 0.5 Hz.
example_cycle_sr = example['sr']

# Concatenate the cleaned cycles chronologically. This omits the beginning
# before cycle_start_samples and the incomplete cycle at the end.
clean_y = np.asarray(example_cycle_sr.stimulus).reshape(-1, order='F')
clean_x = np.asarray(example_cycle_sr.response).reshape(-1, order='F')
example_clean_sr = bp.data_class.sr_data(
    samplingrate_Hz=example_cycle_sr.samplingrate_Hz,
    stimulus=clean_y,
    response=clean_x,
    frequency_selection='all',
    name=f"{example['subject_id']}_{example['condition']} cleaned trial",
)
cycle_freq = example_cycle_sr.freq
cycle_sy = example_cycle_sr.stimulus_spectrum
cycle_sx = example_cycle_sr.response_spectrum

fig_spectra = make_subplots(
    rows=2, cols=1, subplot_titles=['Sᵧ', 'Sₓ'],
    shared_xaxes=True, vertical_spacing=0.14,
)

def add_qualitycheck_spectrum(fig, spectrum, mean_color, mean_label, row, col):
    # Individual-cycle amplitudes: grey markers, exactly as in plot_datacheck.
    for cycle_index in range(spectrum.shape[1]):
        fig.add_trace(
            go.Scatter(
                x=cycle_freq, y=np.abs(spectrum[:, cycle_index]),
                mode='markers', marker=dict(size=3, color='grey'),
                opacity=0.5, name='Individual cycles',
                showlegend=(cycle_index == 0 and row == 1 and col == 1),
            ),
            row=row, col=col,
        )

    mean_amplitude = np.abs(np.mean(spectrum, axis=1))
    stem_x = np.repeat(cycle_freq, 3)
    stem_y = np.empty(len(cycle_freq) * 3)
    stem_y[0::3] = 0
    stem_y[1::3] = mean_amplitude
    stem_y[2::3] = np.nan
    fig.add_trace(
        go.Scatter(
            x=stem_x, y=stem_y, mode='lines',
            line=dict(width=1.5, color=mean_color), showlegend=False,
        ),
        row=row, col=col,
    )
    fig.add_trace(
        go.Scatter(
            x=cycle_freq, y=mean_amplitude, mode='markers',
            marker=dict(size=5, color=mean_color),
            name=mean_label, showlegend=(mean_label != 'Mean stimulus'),
        ),
        row=row, col=col,
    )

add_qualitycheck_spectrum(
    fig_spectra, cycle_sy, 'darkblue', 'Mean stimulus', row=1, col=1
)
add_qualitycheck_spectrum(
    fig_spectra, cycle_sx, 'darkgreen', 'Mean sway response', row=2, col=1
)

for row in (1, 2):
    fig_spectra.update_xaxes(
        title_text='f (Hz)', tickmode='array',
        tickvals=np.arange(0, 2.01, 0.5), range=[0, 2.05], row=row, col=1,
    )
    fig_spectra.update_yaxes(title_text='amplitude (°)', row=row, col=1)
# Hide only the lower graph's x-axis label at zero to avoid two adjacent zeros.
fig_spectra.update_xaxes(
    ticktext=['', '0.5', '1', '1.5', '2'], row=2, col=1,
)
fig_spectra.update_xaxes(
    tickfont=dict(family='Times New Roman', size=18),
    title_font=dict(family='Times New Roman', size=22),
)
fig_spectra.update_yaxes(
    tickfont=dict(family='Times New Roman', size=18),
    title_font=dict(family='Times New Roman', size=22),
)
for annotation in fig_spectra.layout.annotations:
    if annotation.text in {'Sᵧ', 'Sₓ'}:
        annotation.font = dict(family='Times New Roman', size=26)
fig_spectra.update_layout(
    template='plotly_white', height=1000, width=1000,
    font=dict(family='Times New Roman', size=20),
    legend=dict(
        x=1.02, y=1.0, xanchor='left', yanchor='top',
        bgcolor='rgba(255,255,255,0.85)', bordercolor='lightgrey', borderwidth=1,
    ),
    margin=dict(l=110, r=220, t=110, b=90),
)
fig_spectra.add_annotation(
    text='0.1 °/s', x=0, y=1.04, xref='paper', yref='paper',
    showarrow=False, xanchor='left', font=dict(family='Times New Roman', size=20),
)
fig_spectra.show()

# Time signals for the selected trial

Use the same participant and trial as above. The upper panel shows the stimulus and the lower panel shows the sway response. Cleaned cycles are concatenated, excluding the beginning before the first complete cycle and the incomplete final cycle.


In [5]:
example_y = example_clean_sr.stimulus
example_x = example_clean_sr.response

fig_example = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.10)
fig_example.add_trace(go.Scatter(x=example_clean_sr.time, y=example_y, mode='lines', showlegend=False), row=1, col=1)
fig_example.add_trace(go.Scatter(x=example_clean_sr.time, y=example_x, mode='lines', showlegend=False), row=2, col=1)
fig_example.update_yaxes(title_text='Visual scene tilt (°)', row=1, col=1)
fig_example.update_yaxes(title_text='Sway response (°)', row=2, col=1)
fig_example.update_xaxes(title_text='Time (s)', row=2, col=1)
fig_example.update_layout(
    template='plotly_white', height=700, width=1100,
    font=dict(family='Times New Roman', size=20),
    margin=dict(l=100, r=40, t=70, b=80),
)
fig_example.add_annotation(
    text='0.1 °/s', x=0.99, y=1.04, xref='paper', yref='paper',
    showarrow=False, xanchor='right', font=dict(family='Times New Roman', size=20),
)
fig_example.update_xaxes(
    tickfont=dict(family='Times New Roman', size=18),
    title_font=dict(family='Times New Roman', size=22),
)
fig_example.update_yaxes(
    tickfont=dict(family='Times New Roman', size=18),
    title_font=dict(family='Times New Roman', size=22),
)
fig_example.show()

# Mean time signals and amplitude spectra

Pool recordings across participants for each condition. Condition labels appear to the left of each row; bars denote pooled means.


# Combined results grid


In [6]:
column_titles = ['y̅', 'x̅', 'S̅ᵧ', 'S̅ₓ']
fig_grid = make_subplots(
    rows=len(condition_results), cols=4,
    column_titles=column_titles,
    horizontal_spacing=0.055, vertical_spacing=0.055,
)

for row, result in enumerate(condition_results, start=1):
    group = result['group']
    y_bar = group.stimulus_mean
    x_bar = group.response_mean0
    sy_bar = np.abs(group.stimulus_spectrum_mean)
    sx_bar = np.abs(group.response_spectrum_mean)

    # Time-domain means are continuous signals.
    fig_grid.add_trace(
        go.Scatter(x=group.time, y=y_bar, mode='lines', showlegend=False),
        row=row, col=1,
    )
    fig_grid.add_trace(
        go.Scatter(x=group.time, y=x_bar, mode='lines', showlegend=False),
        row=row, col=2,
    )

    # Frequency-domain means are discrete values, so draw stems rather than
    # connecting adjacent frequency bins with a continuous line.
    for col, amplitude, color in ((3, sy_bar, 'darkblue'), (4, sx_bar, 'darkgreen')):
        stem_x = np.repeat(group.freq, 3)
        stem_y = np.empty(len(group.freq) * 3)
        stem_y[0::3] = 0
        stem_y[1::3] = amplitude
        stem_y[2::3] = np.nan
        fig_grid.add_trace(
            go.Scatter(
                x=stem_x, y=stem_y, mode='lines',
                line=dict(width=1.5, color=color), showlegend=False,
            ),
            row=row, col=col,
        )
        fig_grid.add_trace(
            go.Scatter(
                x=group.freq, y=amplitude, mode='markers',
                marker=dict(size=5, color=color), showlegend=False,
            ),
            row=row, col=col,
        )

    first_axis_number = 4 * (row - 1) + 1
    axis_suffix = '' if first_axis_number == 1 else str(first_axis_number)
    fig_grid.add_annotation(
        x=-0.075, y=0.5, xref=f'x{axis_suffix} domain',
        yref=f'y{axis_suffix} domain',
        text=f"<b>{result['label']}</b>",
        showarrow=False, xanchor='right', yanchor='middle',
    )

for col in (1, 2):
    fig_grid.update_xaxes(title_text='Time (s)', row=len(condition_results), col=col)
for row in range(1, len(condition_results) + 1):
    for col in (3, 4):
        fig_grid.update_xaxes(
            tickmode='array', tickvals=np.arange(0, 2.01, 0.5),
            range=[0, 2.05], row=row, col=col,
        )
for col in (3, 4):
    fig_grid.update_xaxes(title_text='Frequency (Hz)', row=len(condition_results), col=col)

fig_grid.update_layout(
    title='Pooled mean time signals and amplitude spectra by condition',
    template='plotly_white',
    height=1250, width=1450,
    margin=dict(l=120, r=30, t=100, b=70),
)


# Build mean time-signal and amplitude-spectrum figures


In [7]:
# Publication figure 1: pooled mean time-domain signals.
fig_grid_time = make_subplots(
    rows=len(condition_results), cols=2,
    column_titles=['y̅', 'x̅'],
    shared_xaxes=True, column_widths=[0.5, 0.5],
    horizontal_spacing=0.18, vertical_spacing=0.07,
)

# Publication figure 2: pooled mean amplitude spectra.
fig_grid_frequency = make_subplots(
    rows=len(condition_results), cols=2,
    column_titles=['S̅ᵧ', 'S̅ₓ'],
    shared_xaxes=True, column_widths=[0.5, 0.5],
    horizontal_spacing=0.18, vertical_spacing=0.07,
)

for row, result in enumerate(condition_results, start=1):
    group = result['group']
    y_bar = group.stimulus_mean
    x_bar = group.response_mean0
    sy_bar = np.abs(group.stimulus_spectrum_mean)
    sx_bar = np.abs(group.response_spectrum_mean)

    fig_grid_time.add_trace(
        go.Scatter(
            x=group.time, y=y_bar, mode='lines',
            line=dict(width=2.5, color='darkblue'), showlegend=False,
        ),
        row=row, col=1,
    )
    fig_grid_time.add_trace(
        go.Scatter(
            x=group.time, y=x_bar, mode='lines',
            line=dict(width=2.5, color='darkgreen'), showlegend=False,
        ),
        row=row, col=2,
    )

    for col, amplitude, color in ((1, sy_bar, 'darkblue'), (2, sx_bar, 'darkgreen')):
        amplitude_max = float(np.max(amplitude))
        amplitude_upper = amplitude_max * 1.08 if amplitude_max > 0 else 1e-6
        stem_start = np.minimum(amplitude, amplitude_upper * 0.003)
        stem_x = np.repeat(group.freq, 3)
        stem_y = np.empty(len(group.freq) * 3)
        stem_y[0::3] = stem_start
        stem_y[1::3] = amplitude
        stem_y[2::3] = np.nan
        fig_grid_frequency.add_trace(
            go.Scatter(
                x=stem_x, y=stem_y, mode='lines',
                line=dict(width=2.5, color=color), showlegend=False,
            ),
            row=row, col=col,
        )
        fig_grid_frequency.update_yaxes(
            range=[0, amplitude_upper], row=row, col=col,
        )
        fig_grid_frequency.add_trace(
            go.Scatter(
                x=group.freq, y=amplitude, mode='markers',
                marker=dict(size=8, color=color), showlegend=False,
            ),
            row=row, col=col,
        )

    # Put the condition at the left of each figure row.
    first_axis_number = 2 * (row - 1) + 1
    yaxis_name = 'yaxis' if first_axis_number == 1 else f'yaxis{first_axis_number}'
    row_midpoint = float(np.mean(fig_grid_time.layout[yaxis_name].domain))
    condition_text = (
        'No stim' if result['velocity'] is None else f"{result['label']} °/s"
    )
    for figure, label_x in (
        (fig_grid_time, -0.20), (fig_grid_frequency, -0.20)
    ):
        figure.add_annotation(
            x=label_x, y=row_midpoint, xref='paper', yref='paper',
            text=f"<b>{condition_text}</b>",
            font=dict(family='Times New Roman', size=20),
            width=140, align='center', showarrow=False,
            xanchor='center', yanchor='middle',
        )



# Format mean time-signal and amplitude-spectrum figures


In [8]:
middle_row = (len(condition_results) + 1) // 2
fig_grid_time.update_yaxes(
    title_text='Screen tilt (°)', title_standoff=14,
    title_font=dict(family='Times New Roman', size=20), row=middle_row, col=1,
)
fig_grid_time.update_yaxes(
    title_text='Sway response (°)', title_standoff=14,
    title_font=dict(family='Times New Roman', size=20), row=middle_row, col=2,
)
fig_grid_frequency.update_yaxes(
    title_text='Screen tilt amplitude (°)', title_standoff=14,
    title_font=dict(family='Times New Roman', size=20), row=middle_row, col=1,
)
fig_grid_frequency.update_yaxes(
    title_text='Sway response amplitude (°)', title_standoff=14,
    title_font=dict(family='Times New Roman', size=20), row=middle_row, col=2,
)

for col in (1, 2):
    fig_grid_time.update_xaxes(
        title_text='Time (s)',
        title_font=dict(family='Times New Roman', size=22),
        tickfont=dict(family='Times New Roman', size=18),
        row=len(condition_results), col=col,
    )
    fig_grid_frequency.update_xaxes(
        title_text='Frequency (Hz)',
        title_font=dict(family='Times New Roman', size=22),
        tickfont=dict(family='Times New Roman', size=18),
        tickmode='array', tickvals=np.arange(0.5, 2.01, 0.5), range=[0, 2.05],
        row=len(condition_results), col=col,
    )

for row in range(1, len(condition_results) + 1):
    for col in (1, 2):
        fig_grid_frequency.update_xaxes(
            tickmode='array', tickvals=np.arange(0.5, 2.01, 0.5),
            range=[0, 2.05], row=row, col=col,
        )

axis_style = dict(
    tickfont=dict(family='Times New Roman', size=18),
    automargin=True, showline=True,
    linecolor='black', linewidth=1, ticks='outside',
)
fig_grid_time.update_xaxes(**axis_style)
fig_grid_time.update_yaxes(**axis_style)
fig_grid_frequency.update_xaxes(**axis_style)
fig_grid_frequency.update_yaxes(
    **axis_style, zeroline=True,
    zerolinecolor='black', zerolinewidth=1,
)
# Use a readable fixed scale only for the first (No stim, S-bar-y) graph.
fig_grid_frequency.update_yaxes(
    range=[0, 1], tickmode='array', tickvals=np.arange(0, 1.01, 0.2),
    tickformat='~g', row=1, col=1,
)
fig_grid_frequency.update_xaxes(ticks='', ticklen=0)

# Enlarge the two column headings without changing condition annotations.
for figure, headings in (
    (fig_grid_time, {'y̅', 'x̅'}),
    (fig_grid_frequency, {'S̅ᵧ', 'S̅ₓ'}),
):
    for annotation in figure.layout.annotations:
        if annotation.text in headings:
            annotation.font = dict(family='Times New Roman', size=26)

for figure in (fig_grid_time, fig_grid_frequency):
    figure.add_annotation(
        text='<b>Condition</b>', x=-0.20, y=1.04,
        xref='paper', yref='paper', showarrow=False,
        xanchor='center', yanchor='bottom', align='center', width=140,
        font=dict(family='Times New Roman', size=22),
    )
fig_grid_time.update_layout(
    template='plotly_white', height=1500, width=1400,
    font=dict(family='Times New Roman', size=20),
    margin=dict(l=340, r=80, t=130, b=100),
)
fig_grid_frequency.update_layout(
    template='plotly_white', height=1500, width=1400,
    font=dict(family='Times New Roman', size=20),
    margin=dict(l=340, r=80, t=130, b=100),
);


# Plot mean time signals


In [9]:
fig_grid_time.show()


# Plot mean amplitude spectra


In [10]:
fig_grid_frequency.show()


# Amplitudes at 0.5 Hz

Tabulate the magnitudes of the pooled mean complex stimulus and response spectra at the stimulation frequency.


In [11]:
main_frequency_rows = []
for result in condition_results:
    group = result['group']
    frequency_index = int(np.argmin(np.abs(group.freq - MAIN_FREQUENCY_HZ)))
    actual_frequency = float(group.freq[frequency_index])
    if not np.isclose(actual_frequency, MAIN_FREQUENCY_HZ):
        raise ValueError(
            f'Expected {MAIN_FREQUENCY_HZ} Hz but nearest bin is {actual_frequency} Hz.'
        )
    main_frequency_rows.append({
        'condition': result['label'],
        'participants': result['n_participants'],
        "f' (Hz)": actual_frequency,
        "mean Sy(f')": np.abs(group.stimulus_spectrum_mean[frequency_index]),
        "mean Sx(f')": np.abs(group.response_spectrum_mean[frequency_index]),
    })

main_frequency_table = pd.DataFrame(main_frequency_rows)
main_frequency_table

,condition,participants,f' (Hz),mean Sy(f'),mean Sx(f')
0,No stim,13,0.5,0.000000,0.001979
1,0.01,13,0.5,0.004053,0.002352
2,0.1,12,0.5,0.040529,0.025425
3,1,13,0.5,0.405285,0.037790
4,10,13,0.5,4.052798,0.034389


# Save thesis figures

Save the four displayed figures to `results/paper_figures` as interactive HTML files. Export SVG and high-resolution PNG copies when static export is available.


In [12]:
# Export the exact Plotly figure objects displayed in this notebook.
paper_figure_dir = paths.project_root / 'results' / 'paper_figures'
paper_figure_dir.mkdir(parents=True, exist_ok=True)
paper_figures = {
    'figure_spectra': fig_spectra,
    'figure_time_domain': fig_example,
    'figure_results_time_domain': fig_grid_time,
    'figure_results_frequency_domain': fig_grid_frequency,
}

# HTML needs no Kaleido and preserves the figures exactly.
for filename, figure in paper_figures.items():
    figure.write_html(
        paper_figure_dir / f'{filename}.html',
        include_plotlyjs=True, full_html=True,
    )

# Static export is optional: use it only when Kaleido is available.
static_export_error = None
try:
    for filename, figure in paper_figures.items():
        figure.write_image(paper_figure_dir / f'{filename}.svg')
        figure.write_image(paper_figure_dir / f'{filename}.png', scale=3)
except Exception as exc:
    static_export_error = exc

print(f'Exact Plotly figures saved as HTML to: {paper_figure_dir}')
if static_export_error is not None:
    print(f'PNG/SVG export could not complete: {static_export_error}')
    print('Open an HTML file and click the camera icon to download the exact PNG.')
sorted(path.name for path in paper_figure_dir.iterdir())

['figure_11.svg',
 'figure_9.png',
 'figure_mean_participant_gain_t_ci.html',
 'figure_mean_participant_gain_t_ci.png',
 'figure_mean_participant_gain_t_ci.svg',
 'figure_results_frequency_domain.html',
 'figure_results_frequency_domain.png',
 'figure_results_frequency_domain.svg',
 'figure_results_time_domain.html',
 'figure_results_time_domain.png',
 'figure_results_time_domain.svg',
 'figure_session_effect_gain_0p01.html',
 'figure_session_effect_gain_0p01.png',
 'figure_spectra.html',
 'figure_spectra.png',
 'figure_spectra.svg',
 'figure_stimulus_amplitude_by_velocity.html',
 'figure_stimulus_amplitude_by_velocity.png',
 'figure_stimulus_amplitude_by_velocity.svg',
 'figure_sway_response_by_velocity.html',
 'figure_sway_response_by_velocity.png',
 'figure_time_domain.html',
 'figure_time_domain.png',
 'figure_time_domain.svg']